# Outposts Graph: Connecting Remote Objects

This notebook demonstrates how to connect remote (non-adjacent) topologies using an "outposts" relationship graph.

**Adapted from topologicpy to use topologic_fast**

Typically, topological graphs only connect adjacent elements (cells sharing a face, faces sharing an edge, etc.). However, sometimes we need to establish relationships between objects that are not physically adjacent - this is where "outposts" come in.

Key concepts:
1. Create a CellComplex with multiple cells
2. Define outpost relationships (which cells should connect to which)
3. Build a graph based on these relationships
4. Visualize the result with Plotly

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import random
from collections import defaultdict

## 1. Create a 3x3x3 Grid of Cells

We'll create a CellComplex similar to a Rubik's cube - 27 cells arranged in a 3x3x3 grid.

In [ ]:
# Create individual cells in a 3x3x3 grid
cells = []
cell_ids = {}  # (i, j, k) -> cell_id
cell_centroids = {}  # cell_id -> (x, y, z)

cell_size = 1.0
gap = 0.1  # Small gap between cells
spacing = cell_size + gap

cell_id = 1
for k in range(3):  # z
    for j in range(3):  # y
        for i in range(3):  # x
            x = i * spacing
            y = j * spacing
            z = k * spacing
            
            cell = tf.Cell.Box(x, y, z, cell_size, cell_size, cell_size)
            cells.append(cell)
            
            cell_ids[(i, j, k)] = cell_id
            # Centroid is at center of cell
            cell_centroids[cell_id] = (x + cell_size/2, y + cell_size/2, z + cell_size/2)
            
            cell_id += 1

print(f"Created {len(cells)} cells in a 3x3x3 grid")
print(f"Cell IDs range from 1 to {len(cells)}")

In [ ]:
# Create CellComplex from all cells
cell_complex = tf.CellComplex.ByCells(cells)

print(f"CellComplex Statistics:")
print(f"  Number of cells: {cell_complex.NumCells()}")
print(f"  Total volume: {cell_complex.Volume():.2f}")
print(f"  Total area: {cell_complex.Area():.2f}")

## 2. Define Adjacency Relationships

We'll find which cells are adjacent (share a face) in the original grid.

**Note:** topologic_fast's CellComplex doesn't expose adjacent cell queries directly, so we calculate adjacency from grid positions.

In [ ]:
def get_adjacent_cells(i, j, k, cell_ids):
    """Get IDs of cells adjacent to position (i, j, k)."""
    adjacents = []
    
    # 6 face-sharing neighbors
    neighbors = [
        (i-1, j, k), (i+1, j, k),  # x direction
        (i, j-1, k), (i, j+1, k),  # y direction
        (i, j, k-1), (i, j, k+1)   # z direction
    ]
    
    for ni, nj, nk in neighbors:
        if (ni, nj, nk) in cell_ids:
            adjacents.append(cell_ids[(ni, nj, nk)])
    
    return adjacents

# Build adjacency dictionary
cell_adjacency = {}

for (i, j, k), cid in cell_ids.items():
    adjacent_ids = get_adjacent_cells(i, j, k, cell_ids)
    cell_adjacency[cid] = adjacent_ids

# Print adjacency for each cell
print("Cell Adjacency (face-sharing neighbors):")
print("-" * 40)
for cid in sorted(cell_adjacency.keys()):
    print(f"Cell {cid:2d}: {cell_adjacency[cid]}")

## 3. Define Outpost Relationships

Now we'll define "outpost" relationships - connections between cells that are NOT adjacent.
These could represent:
- Communication links
- Supply routes
- Logical connections
- Any non-physical relationship

In [ ]:
# For this example, we'll connect cells diagonally through the cube
# These are outpost relationships (not based on physical adjacency)

outpost_relationships = {}

# We'll use the same adjacency as outposts for this example
# In real use, outposts would be defined by some other logic
for cid, adj_list in cell_adjacency.items():
    outpost_relationships[cid] = adj_list.copy()

print("Outpost relationships defined for all cells.")
print(f"Total cells: {len(outpost_relationships)}")
print(f"Total outpost connections: {sum(len(v) for v in outpost_relationships.values()) // 2}")

## 4. Explode and Randomize Cell Positions

We'll move cells to random positions to show that the outpost graph
maintains relationships regardless of physical location.

In [ ]:
# Explode cells outward from center and add random displacement
exploded_cells = []
exploded_centroids = {}  # cell_id -> (x, y, z)

# Center of the grid
center_x = (3 - 1) * spacing / 2
center_y = (3 - 1) * spacing / 2
center_z = (3 - 1) * spacing / 2

scale_factor = 1.5  # How much to explode
random_factor = 0.3  # Random displacement

cell_id = 1
for k in range(3):
    for j in range(3):
        for i in range(3):
            # Original centroid
            orig_x, orig_y, orig_z = cell_centroids[cell_id]
            
            # Vector from center
            dx = orig_x - center_x
            dy = orig_y - center_y
            dz = orig_z - center_z
            
            # Exploded position
            new_x = center_x + dx * scale_factor + random.uniform(-random_factor, random_factor)
            new_y = center_y + dy * scale_factor + random.uniform(-random_factor, random_factor)
            new_z = center_z + dz * scale_factor + random.uniform(-random_factor, random_factor)
            
            # Create new cell at exploded position
            cell = tf.Cell.Box(new_x - cell_size/2, new_y - cell_size/2, new_z - cell_size/2,
                              cell_size, cell_size, cell_size)
            exploded_cells.append(cell)
            exploded_centroids[cell_id] = (new_x, new_y, new_z)
            
            cell_id += 1

print(f"Exploded {len(exploded_cells)} cells with random displacement")

## 5. Build Outposts Graph

Create a graph where:
- Vertices are at cell centroids
- Edges connect cells that have outpost relationships

**Note:** topologic_fast's `Graph.ByTopology()` doesn't support the `toOutposts` parameter, so we build the graph manually.

In [ ]:
# Create vertices at exploded cell centroids
graph_vertices = []
vertex_by_id = {}  # cell_id -> vertex

for cell_id in range(1, len(exploded_cells) + 1):
    x, y, z = exploded_centroids[cell_id]
    v = tf.Vertex.ByCoordinates(x, y, z)
    graph_vertices.append(v)
    vertex_by_id[cell_id] = v

print(f"Created {len(graph_vertices)} graph vertices")

In [ ]:
# Create edges based on outpost relationships
graph_edges = []
edge_set = set()  # Track added edges to avoid duplicates

for cell_id, outposts in outpost_relationships.items():
    for outpost_id in outposts:
        # Avoid duplicate edges (1->2 and 2->1)
        edge_key = tuple(sorted([cell_id, outpost_id]))
        if edge_key not in edge_set:
            edge_set.add(edge_key)
            
            v1 = vertex_by_id[cell_id]
            v2 = vertex_by_id[outpost_id]
            edge = tf.Edge.ByStartVertexEndVertex(v1, v2)
            graph_edges.append(edge)

print(f"Created {len(graph_edges)} graph edges")

In [ ]:
# Build the graph
outpost_graph = tf.Graph.ByVerticesEdges(graph_vertices, graph_edges)

print(f"Outpost Graph Statistics:")
print(f"  Vertices: {outpost_graph.Order()}")
print(f"  Edges: {outpost_graph.Size()}")
print(f"  Density: {outpost_graph.Density():.4f}")
print(f"  Diameter: {outpost_graph.Diameter()} steps")
print(f"  Is Complete: {outpost_graph.IsComplete()}")

## 6. Visualize the Exploded Cells with Outpost Graph

In [ ]:
def visualize_outpost_graph(cells, graph, cell_centroids):
    """Create 3D visualization of cells with outpost graph overlay."""
    fig = go.Figure()
    
    # Draw cells
    for i, cell in enumerate(cells):
        faces = cell.Faces()
        
        # Random color for each cell
        color = f'rgba({random.randint(100,200)}, {random.randint(100,200)}, {random.randint(100,200)}, 0.3)'
        
        for face in faces:
            vertices = face.Vertices()
            coords = [v.Coordinates() for v in vertices]
            
            if len(coords) >= 3:
                x = [c[0] for c in coords]
                y = [c[1] for c in coords]
                z = [c[2] for c in coords]
                
                fig.add_trace(go.Mesh3d(
                    x=x, y=y, z=z,
                    color=color,
                    opacity=0.4,
                    alphahull=0,
                    name=f'Cell {i+1}',
                    showlegend=False
                ))
    
    # Draw graph edges (outpost connections)
    graph_edges = graph.Edges()
    for edge in graph_edges:
        start, end = edge.Vertices()
        start_c = start.Coordinates()
        end_c = end.Coordinates()
        
        fig.add_trace(go.Scatter3d(
            x=[start_c[0], end_c[0]],
            y=[start_c[1], end_c[1]],
            z=[start_c[2], end_c[2]],
            mode='lines',
            line=dict(color='red', width=4),
            showlegend=False,
            hoverinfo='skip'
        ))
    
    # Draw graph vertices
    graph_verts = graph.Vertices()
    x = [v.Coordinates()[0] for v in graph_verts]
    y = [v.Coordinates()[1] for v in graph_verts]
    z = [v.Coordinates()[2] for v in graph_verts]
    
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers+text',
        marker=dict(size=10, color='blue', symbol='circle'),
        text=[str(i+1) for i in range(len(graph_verts))],
        textposition='top center',
        textfont=dict(size=10, color='darkblue'),
        name='Cell Centroids'
    ))
    
    fig.update_layout(
        title='Exploded CellComplex with Outpost Graph<br><sub>Red edges show outpost (relationship) connections</sub>',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data',
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))
        ),
        width=900,
        height=700,
        showlegend=True
    )
    
    return fig

fig_3d = visualize_outpost_graph(exploded_cells, outpost_graph, exploded_centroids)
fig_3d.show()

## 7. Connectivity Analysis

In [ ]:
# Analyze connectivity of each cell
print("Cell Connectivity Analysis:")
print("=" * 50)

connectivity_data = []
graph_verts = outpost_graph.Vertices()

for i, v in enumerate(graph_verts):
    cell_id = i + 1
    degree = outpost_graph.VertexDegree(v)
    
    # Get position in original grid
    grid_pos = None
    for pos, cid in cell_ids.items():
        if cid == cell_id:
            grid_pos = pos
            break
    
    connectivity_data.append({
        'cell_id': cell_id,
        'grid_pos': grid_pos,
        'degree': degree,
        'outposts': outpost_relationships.get(cell_id, [])
    })

# Sort by degree
connectivity_data.sort(key=lambda x: -x['degree'])

for data in connectivity_data:
    pos_str = f"({data['grid_pos'][0]},{data['grid_pos'][1]},{data['grid_pos'][2]})"
    print(f"Cell {data['cell_id']:2d} at {pos_str}: {data['degree']} connections -> {data['outposts']}")

In [ ]:
# Degree distribution
degree_dist = defaultdict(int)
for data in connectivity_data:
    degree_dist[data['degree']] += 1

print("\nDegree Distribution:")
print("-" * 30)
for deg in sorted(degree_dist.keys()):
    print(f"  Degree {deg}: {degree_dist[deg]} cells")

## 8. Shortest Path in Outpost Graph

In [ ]:
# Find paths between corner cells
# Corner cells are at positions (0,0,0) and (2,2,2)

corner_1 = cell_ids[(0, 0, 0)]  # Bottom-front-left
corner_2 = cell_ids[(2, 2, 2)]  # Top-back-right

v1 = vertex_by_id[corner_1]
v2 = vertex_by_id[corner_2]

distance = outpost_graph.Distance(v1, v2)
path = outpost_graph.Path(v1, v2)

print(f"Shortest path from Cell {corner_1} to Cell {corner_2}:")
print(f"  Distance: {distance} hops")

if path:
    path_verts = path.Vertices()
    path_cells = []
    
    for pv in path_verts:
        pv_coords = pv.Coordinates()
        # Find matching cell
        for cell_id, coords in exploded_centroids.items():
            if (abs(pv_coords[0] - coords[0]) < 0.01 and 
                abs(pv_coords[1] - coords[1]) < 0.01 and
                abs(pv_coords[2] - coords[2]) < 0.01):
                path_cells.append(cell_id)
                break
    
    print(f"  Path: {' -> '.join(map(str, path_cells))}")

In [ ]:
def visualize_path(cells, graph, path, cell_centroids):
    """Visualize a specific path through the outpost graph."""
    fig = go.Figure()
    
    # Draw cells (faded)
    for cell in cells:
        faces = cell.Faces()
        for face in faces:
            vertices = face.Vertices()
            coords = [v.Coordinates() for v in vertices]
            
            if len(coords) >= 3:
                x = [c[0] for c in coords]
                y = [c[1] for c in coords]
                z = [c[2] for c in coords]
                
                fig.add_trace(go.Mesh3d(
                    x=x, y=y, z=z,
                    color='lightgray',
                    opacity=0.2,
                    alphahull=0,
                    showlegend=False
                ))
    
    # Draw all graph edges (faded)
    graph_edges = graph.Edges()
    for edge in graph_edges:
        start, end = edge.Vertices()
        start_c = start.Coordinates()
        end_c = end.Coordinates()
        
        fig.add_trace(go.Scatter3d(
            x=[start_c[0], end_c[0]],
            y=[start_c[1], end_c[1]],
            z=[start_c[2], end_c[2]],
            mode='lines',
            line=dict(color='lightgray', width=2),
            showlegend=False,
            hoverinfo='skip'
        ))
    
    # Draw the path
    if path:
        path_verts = path.Vertices()
        x = [v.Coordinates()[0] for v in path_verts]
        y = [v.Coordinates()[1] for v in path_verts]
        z = [v.Coordinates()[2] for v in path_verts]
        
        fig.add_trace(go.Scatter3d(
            x=x, y=y, z=z,
            mode='lines+markers',
            line=dict(color='red', width=8),
            marker=dict(size=15, color='red'),
            name='Shortest Path'
        ))
    
    fig.update_layout(
        title=f'Shortest Path from Cell {corner_1} to Cell {corner_2}',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data',
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))
        ),
        width=900,
        height=700
    )
    
    return fig

fig_path = visualize_path(exploded_cells, outpost_graph, path, exploded_centroids)
fig_path.show()

## 9. Depth Map Visualization

In [ ]:
# Calculate depth map from corner cell
source_cell = cell_ids[(0, 0, 0)]
source_vertex = vertex_by_id[source_cell]

depth_map = outpost_graph.DepthMap(source_vertex)

print(f"Depth Map from Cell {source_cell} (corner):")
print("-" * 40)

for i, depth in enumerate(depth_map):
    cell_id = i + 1
    # Get grid position
    for pos, cid in cell_ids.items():
        if cid == cell_id:
            print(f"  Cell {cell_id:2d} at {pos}: depth = {depth}")
            break

In [ ]:
def visualize_depth_map(cells, graph, depth_map, cell_centroids):
    """Visualize cells colored by depth from source."""
    fig = go.Figure()
    
    # Color scale based on depth
    max_depth = max(depth_map)
    colors = ['green', 'yellow', 'orange', 'red', 'purple', 'blue']
    
    # Draw graph vertices colored by depth
    graph_verts = graph.Vertices()
    
    for i, v in enumerate(graph_verts):
        depth = depth_map[i]
        color = colors[depth % len(colors)]
        
        coords = v.Coordinates()
        fig.add_trace(go.Scatter3d(
            x=[coords[0]],
            y=[coords[1]],
            z=[coords[2]],
            mode='markers+text',
            marker=dict(size=20, color=color),
            text=[f"{i+1} (d={depth})"],
            textposition='top center',
            textfont=dict(size=8),
            showlegend=False,
            hovertext=f"Cell {i+1}, Depth: {depth}"
        ))
    
    # Draw edges
    graph_edges = graph.Edges()
    for edge in graph_edges:
        start, end = edge.Vertices()
        start_c = start.Coordinates()
        end_c = end.Coordinates()
        
        fig.add_trace(go.Scatter3d(
            x=[start_c[0], end_c[0]],
            y=[start_c[1], end_c[1]],
            z=[start_c[2], end_c[2]],
            mode='lines',
            line=dict(color='gray', width=2),
            showlegend=False,
            hoverinfo='skip'
        ))
    
    # Add legend
    for d in range(max_depth + 1):
        fig.add_trace(go.Scatter3d(
            x=[None], y=[None], z=[None],
            mode='markers',
            marker=dict(size=10, color=colors[d % len(colors)]),
            name=f'Depth {d}'
        ))
    
    fig.update_layout(
        title=f'Depth Map from Cell {source_cell} (green = source)',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data',
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))
        ),
        width=900,
        height=700,
        showlegend=True
    )
    
    return fig

fig_depth = visualize_depth_map(exploded_cells, outpost_graph, depth_map, exploded_centroids)
fig_depth.show()

## Summary

This notebook demonstrated:

1. **CellComplex Creation** - Built a 3x3x3 grid of cells using `tf.Cell.Box()` and `tf.CellComplex.ByCells()`
2. **Adjacency Calculation** - Determined which cells share faces
3. **Outpost Relationships** - Defined non-physical connections between cells
4. **Graph Construction** - Built a graph using `tf.Graph.ByVerticesEdges()`
5. **Path Finding** - Found shortest paths using `Graph.Path()`
6. **Depth Analysis** - Calculated depth maps using `Graph.DepthMap()`

### Key Differences from topologicpy:

- **No toOutposts parameter**: topologic_fast's `Graph.ByTopology()` doesn't support outpost relationships directly
- **No idKey/outpostsKey**: We build the graph manually from vertices and edges
- **No Topology.Explode()**: We calculate exploded positions manually
- **No transferDictionaries**: We track cell IDs using Python dictionaries

### Applications:
- **Building Information Modeling**: Connect non-adjacent spaces (elevator access, etc.)
- **Network Design**: Model logical connections independent of physical layout
- **Game Development**: Define "teleport" or "warp" connections between areas
- **Supply Chain**: Model relationships between warehouses, suppliers, customers